# FIX-05 — `Discrete` observation spaces are unusable under SimplyQRL's DQN

**Self-contained.** This notebook needs nothing from `qrl-dissection`. It installs
the pinned stack, reproduces the bug from scratch, and verifies the fix. Section 9
then cross-checks that the repository's `core/obs_adapters.py` gives the same answer.

FrozenLake is the second environment for the dissection, and upstream's DQN cannot
run it at all. Before building an experiment on top, the failure needs reproducing
and understanding — not assuming from a code reading.

**Claim under test.** `gym.spaces.Discrete(n).shape` is `()`, not `(1,)`, and its
dtype is `int64`. SimplyQRL derives every tensor shape from that attribute, so
three things break at once — and the third is silent, which is what makes this a
gate rather than a footnote.

| # | where | symptom |
|---|---|---|
| 1 | `dqn.py` action selection | `IndexError` on the first step |
| 2 | `buffers.py` allocation | batch is `[B]` `int64`, not `[B, 1]` `float` |
| 3 | `transformations.py` | **silent**: one sample's angles for the whole batch |

PPO is unaffected. Section 5 shows why, and that is the interesting part.

---
## 0. Install (pinned stack)

In [ ]:
import sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "gymnasium==1.1.1", "autoray==0.7.1",
                    "pennylane==0.41.1", "pennylane-lightning==0.41.1",
                    "git+https://github.com/javier-lazaro/simplyqrl.git@b534cc9"], check=False)
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "jax", "jaxlib"], check=False)

import numpy as np, gymnasium as gym
print("gymnasium", gym.__version__)
try:
    import torch; print("torch", torch.__version__)
except ImportError:
    torch = None; print("torch NOT available - sections 3, 4, 7, 8 will skip")

---
## 1. The space itself

Everything downstream follows from these two attributes. CartPole is the contrast arm — it is why the bug survived unnoticed.

In [ ]:
FL = dict(map_name="4x4", is_slippery=False)
fl_space = gym.make("FrozenLake-v1", **FL).observation_space
cp_space = gym.make("CartPole-v1").observation_space

print(f"FrozenLake : {fl_space}  shape={fl_space.shape}  dtype={fl_space.dtype}")
print(f"CartPole   : shape={cp_space.shape}  dtype={cp_space.dtype}")

assert fl_space.shape == () and fl_space.dtype == np.int64
print("\nCONFIRMED: Discrete(16).shape is (), so there is no feature axis to inherit.")

---
## 2. Symptom 2 — the buffer allocates without a feature axis

Reproducing `simplyqrl/buffers.py::ReplayBuffer.__init__` verbatim:

```python
obs_shape = observation_space.shape
self.obs_buf = np.zeros((buffer_size, *obs_shape), dtype=observation_space.dtype)
```

In [ ]:
def upstream_allocation(space, buffer_size=1000):
    return np.zeros((buffer_size, *space.shape), dtype=space.dtype)

for name, space in (("FrozenLake", fl_space), ("CartPole", cp_space)):
    buf = upstream_allocation(space)
    batch = buf[np.random.randint(0, 1000, size=128)]
    flag = "   <-- MALFORMED" if batch.ndim == 1 else ""
    print(f"{name:11s} obs_buf {str(buf.shape):12s} {str(buf.dtype):8s} "
          f"-> batch {str(batch.shape):10s} ndim={batch.ndim}{flag}")

assert upstream_allocation(fl_space).ndim == 1
assert upstream_allocation(cp_space).ndim == 2
print("\nCONFIRMED: a 1-D int64 batch where [B, 1] float is expected.")

---
## 3. Symptom 3 — the silent one

The `Frozen*` transformers branch on `data.dim() == 1` to handle a single
observation. Handed a `[B]` batch they take that branch and return **one
sample's angles for the entire batch**. Nothing raises. A run completes and
produces a flat, plausible curve trained on one observation repeated 128 times.

That is why FIX-05 needs a gate in front of every FrozenLake run.

In [ ]:
if torch is None:
    print("skipped (no torch)")
else:
    from simplyqrl.transformations import (FrozenBasisToAngleTransformer,
                                           FrozenNormalizationTransformer)
    t = FrozenBasisToAngleTransformer("4x4")
    good = t(torch.tensor([[3.0], [7.0], [11.0]]))   # [B,1] - correct
    bad  = t(torch.tensor([3.0, 7.0, 11.0]))         # [B]   - what upstream delivers
    print("well-formed [B,1] ->", tuple(good.shape), "(3 distinct samples)")
    print("upstream    [B]   ->", tuple(bad.shape), "(ONE sample, silently)")
    print("\ngood:\n", good.numpy().round(2), "\nbad:\n", bad.numpy().round(2))
    assert good.shape == (3, 4) and bad.shape == (4,)
    print("\nCONFIRMED: the batch collapses to sample 0, no exception raised.")

In [ ]:
if torch is None:
    print("skipped (no torch)")
else:
    # Second silent failure: dtype. FrozenNormalizationTransformer clones the
    # tensor and writes a float into it. On the int64 tensor the buffer hands
    # over, that write truncates - quantising the encoding angle.
    ts = FrozenNormalizationTransformer("4x4")
    as_float = ts(torch.tensor([[5.0]]))
    as_int   = ts(torch.tensor([[5]], dtype=torch.int64))
    print(f"float input -> {as_float.item():.4f} rad")
    print(f"int64 input -> {as_int.item():.4f} rad   <-- truncated")
    assert as_float.item() != as_int.item()
    print("\nCONFIRMED: int64 observations quantise the encoding angle.")

---
## 4. Symptom 1 — action selection crashes

`dqn.py` does `torch.argmax(self.q_network(torch.Tensor(self.obs)), dim=1)`. With
a `Discrete` space `self.obs` has shape `(1,)`, so `nn.Linear` reads it as one
sample with one feature and returns a 1-D `q_values`; `argmax(..., dim=1)` raises.

Unlike sections 2 and 3 this fails **loudly** — which is why nobody ever produced
wrong FrozenLake numbers with this stack. They produced none at all.

In [ ]:
if torch is None:
    print("skipped (no torch)")
else:
    import torch.nn as nn
    qnet = nn.Sequential(nn.Linear(1, 120), nn.ReLU(), nn.Linear(120, 4))
    q = qnet(torch.Tensor(np.array([0])))
    print("q_values from a (1,) observation:", tuple(q.shape))
    try:
        torch.argmax(q, dim=1); print("no error - upstream may have changed")
    except IndexError as e:
        print("IndexError as predicted:", e)
    q2 = qnet(torch.Tensor(np.array([[0.0]])))
    print("\nafter the fix:", tuple(q2.shape), "-> argmax ok:",
          torch.argmax(q2, dim=1).tolist())

---
## 5. Why PPO escapes — the part that matters

`ppo.py` reshapes explicitly, on collection and on the update, and stores
observations in a float tensor. Immediately above the second reshape the
CartPole-only version survives as a comment:

```python
#b_obs = self.obs.reshape((-1,) + self.envs.single_observation_space.shape)
b_obs = self.obs.reshape(self.batch_size, -1)
```

That comment is the clearest evidence yet for the reading FIX-01 already
suggested: **the on-policy path was hardened when the library moved beyond
CartPole, and the off-policy path was never re-validated.** FIX-01 and FIX-05 are
two independent instances of the same history, in two different subsystems.

The cell below reads the installed source rather than trusting the quotation.

In [ ]:
import inspect, re
try:
    import simplyqrl.ppo as _ppo, simplyqrl.dqn as _dqn
    ppo_src, dqn_src = inspect.getsource(_ppo), inspect.getsource(_dqn)
    print("=== ppo.py: obs reshapes ===")
    for line in ppo_src.splitlines():
        if "reshape" in line and "obs" in line: print("   ", line.strip())
    print("\n=== dqn.py: buffer insertion (no reshape anywhere) ===")
    for line in dqn_src.splitlines():
        if "rb.add" in line or "real_next_obs" in line: print("   ", line.strip())
    print("\n=== dqn.py: hardcoded CartPole-shaped diagnostic ===")
    m = re.search(r".*randn\(8, 4\).*", dqn_src)
    print("   ", m.group(0).strip() if m else "(not found)")
    print("    ^ raises on any obs_dim != 4. Diagnostic-only (verbose branch),")
    print("      but a third independent fingerprint of the same history.")
except Exception as e:
    print("could not read installed source:", e)

---
## 6. The fix — an environment adapter, not a patch

We do **not** patch upstream's buffer. We adapt the environment so it presents
its observation in the form the rest of the stack already assumes:
`Box(shape=(1,), float32)`, state index unchanged in value.

Three reasons for this shape of fix:

- it reproduces PPO's effective representation exactly, so a cross-algorithm
  comparison sees the same thing on both sides;
- `SafeDQN` does `gym.make(self.env_id)` and `run_arm`/`run_grid` take an
  `env_id` string, so registering ids means **zero runner changes**;
- it is testable in isolation, which a monkeypatch of upstream internals is not.

In [ ]:
from gymnasium import spaces

class DiscreteToBoxObs(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.n_states = int(env.observation_space.n)
        self.observation_space = spaces.Box(0.0, float(self.n_states-1), (1,), np.float32)
    def observation(self, obs):
        return np.array([obs], dtype=np.float32)

class OneHotObs(gym.ObservationWrapper):
    def __init__(self, env):
        super().__init__(env)
        self.n_states = int(env.observation_space.n)
        self.observation_space = spaces.Box(0.0, 1.0, (self.n_states,), np.float32)
    def observation(self, obs):
        v = np.zeros(self.n_states, dtype=np.float32); v[int(obs)] = 1.0; return v

gym.register(id="FL4x4Scalar-demo", max_episode_steps=100,
             entry_point=lambda **kw: DiscreteToBoxObs(gym.make("FrozenLake-v1", **FL, **kw)))
gym.register(id="FL4x4OneHot-demo", max_episode_steps=100,
             entry_point=lambda **kw: OneHotObs(gym.make("FrozenLake-v1", **FL, **kw)))
print("registered demo ids.")

### The gate

Every assertion that must hold before any FrozenLake training run.

In [ ]:
GATE = []
def gate(label):
    def wrap(fn):
        try:
            fn(); GATE.append((label, None)); print(f"  PASS  {label}")
        except AssertionError as e:
            GATE.append((label, e)); print(f"  FAIL  {label}: {e}")
        return fn
    return wrap

env = gym.make("FL4x4Scalar-demo")

@gate("observation_space is Box(1,) float32")
def _():
    assert env.observation_space.shape == (1,), env.observation_space.shape
    assert env.observation_space.dtype == np.float32

@gate("reset returns (1,) float32")
def _():
    o, _i = env.reset(seed=0); assert o.shape == (1,) and o.dtype == np.float32

@gate("step returns (1,) float32")
def _():
    env.reset(seed=0); o = env.step(env.action_space.sample())[0]
    assert o.shape == (1,) and o.dtype == np.float32

@gate("state value preserved (start cell is 0)")
def _():
    o, _i = env.reset(seed=0); assert float(o[0]) == 0.0, o

@gate("obs_buf is (N, 1) float32")
def _():
    b = upstream_allocation(env.observation_space)
    assert b.shape == (1000, 1) and b.dtype == np.float32, (b.shape, b.dtype)

@gate("sampled batch is 2-D")
def _():
    assert upstream_allocation(env.observation_space)[np.arange(128)].ndim == 2

@gate("TimeLimit pinned at 100 steps")
def _():
    assert env.spec.max_episode_steps == 100, env.spec.max_episode_steps

@gate("one-hot variant is (16,)")
def _():
    assert gym.make("FL4x4OneHot-demo").observation_space.shape == (16,)

fails = [(l, e) for l, e in GATE if e is not None]
print(f"\n{len(GATE)-len(fails)} passed, {len(fails)} failed")
assert not fails, "FIX-05 gate FAILED - do not train until this passes."
print("FIX-05 GATE PASSED")

---
## 7. End to end through the real agent

Shapes are one thing; a real forward pass is another. This builds the chapter's
two FrozenLake configurations and pushes a batch through the Q-network.

Note `ent=False` at one qubit. Not a preference: `ent=True` would build
`CZ(wires=[0, 0])`, a control equal to its target. Config A is therefore
*necessarily* unentangled, which means Config A vs Config B confounds embedding
with entanglement — a point the chapter does not flag.

In [ ]:
if torch is None:
    print("skipped (no torch)")
else:
    from simplyqrl.agents import build_agent
    CONFIGS = {
        "A scalar-to-phase (1q)": dict(circ_type="skolik", n_qubits=1, n_layers_q=5,
                                       ent=False, net_arch=[],
                                       transform_fn=FrozenNormalizationTransformer("4x4")),
        "B binary-basis (4q)":    dict(circ_type="skolik", n_qubits=4, n_layers_q=5,
                                       ent=True, net_arch=[],
                                       transform_fn=FrozenBasisToAngleTransformer("4x4")),
    }
    env = gym.make("FL4x4Scalar-demo")
    for label, cfg in CONFIGS.items():
        net = build_agent("hybrid", env.observation_space.shape, env.action_space.n,
                          config=cfg, is_qnet=True)
        n = sum(p.numel() for p in net.parameters())
        q = net(torch.rand(32, 1) * 15.0)
        single = net(torch.tensor([[0.0]]))
        assert q.shape == (32, 4) and single.shape == (1, 4)
        assert torch.isfinite(q).all()
        assert not torch.allclose(q[0], q[1]), f"{label}: batch collapsed!"
        print(f"  PASS  {label:24s} {n:4d} params   batch {tuple(q.shape)}")
    print("\nBoth configurations forward correctly on a real batch.")

---
## 8. Smoke run and the phantom fraction

Shapes can be right while training still fails. This runs the one-hot classical
arm briefly through the real upstream `DQN` and measures the quantity exp04's
H3 rests on.

Uses upstream `DQN` directly so the notebook stays repo-independent — therefore
**without** FIX-01. Fine here: the only question is whether the loop runs on a
`Discrete` environment at all, and what the phantom fraction looks like.

Reference measured while integrating exp04, on the repo's `SafeDQN` at 6k steps:
**8.6% (FIX-01 off), 9.9% (on)**. Random-policy prediction: mean episode length
7.69 → **13%**. CartPole at convergence: **under 1%**.

In [ ]:
if torch is None:
    print("skipped (no torch)")
else:
    import os, tempfile, pandas as pd
    from simplyqrl.dqn import DQN
    workdir = tempfile.mkdtemp(); cwd = os.getcwd(); os.chdir(workdir)
    try:
        agent = DQN(env=gym.make("FL4x4OneHot-demo"), agent_type="mlp",
                    agent_config={"net_arch": [64, 64]}, run_name="fix05_smoke", seed=1,
                    batch_size=128, buffer_size=50_000, train_frequency=1,
                    learning_starts=1_000)
        agent.train(total_timesteps=5_000, progress_bar=False)
        df = pd.read_csv(os.path.join(workdir, "runs", "fix05_smoke.csv"))
        mean_len = 5000 / max(len(df), 1)
        print(f"\nepisodes logged : {len(df)}")
        print(f"mean length     : {mean_len:.1f} steps")
        print(f"success rate    : {df.ep_reward.mean():.4f}")
        print(f"phantom fraction ~= 1/mean_length = {1/mean_len:.4f}")
        assert len(df) > 0, "no episodes logged"
        print("\nSMOKE RUN PASSED - the loop runs on a Discrete environment.")
    finally:
        os.chdir(cwd)

---
## 9. Cross-check against the repository

Everything above was written standalone. This confirms `core/obs_adapters.py` in the
repo behaves identically — so the evidence in this notebook is evidence about
the code that will actually run.

In [ ]:
import sys, pathlib, subprocess
CODE = pathlib.Path("/content/qrl-dissection")
if not CODE.exists():
    CODE = pathlib.Path.cwd()
    while CODE != CODE.parent and not (CODE/"src"/"qrl_dissection").exists():
        CODE = CODE.parent
try:
    sys.path.insert(0, str(CODE/"src"))
    from qrl_dissection.core.obs_adapters import (FROZEN_ONEHOT_ID, FROZEN_SCALAR_ID,
                                          register_environments)
    register_environments()
    e = gym.make(FROZEN_SCALAR_ID)
    assert e.observation_space.shape == (1,)
    assert e.observation_space.dtype == np.float32
    assert e.spec.max_episode_steps == 100
    assert gym.make(FROZEN_ONEHOT_ID).observation_space.shape == (16,)
    print("repo core/obs_adapters.py matches this notebook:", FROZEN_SCALAR_ID, FROZEN_ONEHOT_ID)
    r = subprocess.run([sys.executable, "-m", "pytest", "-q",
                        str(CODE/"tests"/"test_frozenlake_envs.py")],
                       capture_output=True, text=True, cwd=str(CODE))
    print(r.stdout[-800:])
except Exception as exc:
    print("repo not found or not importable here:", exc)

---
## What this establishes

1. **The bug is real and reproduced from scratch**, not inferred. Three symptoms,
   two of them silent.
2. **The fix is minimal and verifiable**: an environment adapter, no upstream
   patching, no runner changes.
3. **A third fingerprint of the CartPole-only history** — after FIX-01's missing
   autoreset handling and the commented-out reshape in `ppo.py`, the hardcoded
   `torch.randn(8, 4)` diagnostic in `dqn.py`.
4. **The phantom fraction on FrozenLake is ~10x CartPole's at convergence**, which
   is the quantity exp04's H3 predicts the FIX-01 effect will track.

Record sections 7 and 8 in `docs/RESULTS-LOG.md` under Experiment 04 stage 0
*before* running any grid: a prediction is only worth something if it is on record
before the measurement exists.

The same checks live in `tests/test_frozenlake_envs.py` for CI, where they also
**pin the upstream bug**: if a `test_upstream_*` case ever fails, upstream has
repaired it and `core/obs_adapters.py` may be removable.